# Prueba de exploración — turismo

Este notebook cubre las tablas T1–T9 del reto. Los CSV originales deben estar en `data/raw/`.

> Importante: los visitantes son **personas únicas** y no se deben sumar entre días, barrios o desgloses. Los valores `NA` pueden representar anonimización por el umbral de privacidad.

## 1. Preparación

In [ ]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True)

print(f"Proyecto: {ROOT.resolve()}")
print(f"Entrada:  {RAW.resolve()}")


In [ ]:
def find_csv(stem):
    """Busca un CSV por el nombre esperado, permitiendo sufijos y mayúsculas."""
    candidates = sorted(RAW.glob("*.csv"))
    exact = [p for p in candidates if p.stem.lower() == stem.lower()]
    matches = exact or [p for p in candidates if stem.lower() in p.stem.lower()]
    if not matches:
        raise FileNotFoundError(
            f"No se encontró un CSV para '{stem}' en {RAW}. "
            "Copia primero los datos originales en data/raw/."
        )
    if len(matches) > 1:
        print("Aviso: se encontraron varias coincidencias; se usa", matches[0].name)
    return matches[0]

def read_csv(stem):
    path = find_csv(stem)
    df = pd.read_csv(path, low_memory=False)
    df.columns = (
        pd.Index(df.columns).astype(str).str.strip().str.lower()
        .str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("ascii")
        .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")
    )
    print(f"{path.name}: {df.shape[0]:,} filas x {df.shape[1]} columnas")
    return df

def profile(df):
    return (pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "nulos_pct": (df.isna().mean() * 100).round(2),
        "unicos": df.nunique(dropna=True),
    }).sort_values(["nulos_pct", "unicos"], ascending=[False, True]))

def as_date(df, column="day"):
    if column in df:
        df[column] = pd.to_datetime(df[column], errors="coerce")
    return df


## 2. Carga de tablas T1–T9

In [ ]:
files = {
    "T1": "visitantes_unicos_municipio_yearmon",
    "T2": "visitantes_unicos_municipio_date_dt",
    "T3": "pernoctaciones_semana",
    "T4": "visitantes_unicos_barrio_yearmon",
    "T5": "visitantes_unicos_barrio_date_dt",
    "T6": "visitantes_regionales_barrio_yearmon",
    "T7": "visitantes_nacionales_barrio_yearmon",
    "T8": "visitantes_internacionales_barrio_yearmon",
    "T9": "visitantes_sociodemografico_barrio_yearmon",
}

tables = {}
for key, stem in files.items():
    try:
        tables[key] = read_csv(stem)
    except FileNotFoundError as exc:
        print(f"{key}: pendiente — {exc}")

for key, df in tables.items():
    print(f"\n--- {key} ---")
    display(profile(df))


## 3. Validación de columnas y fechas

In [ ]:
required = {
    "T1": {"ano", "mes", "origen", "tipo_visitante", "visitantes"},
    "T2": {"ano", "mes", "date_dt", "origen", "tipo_visitante", "visitantes"},
    "T3": {"ano", "mes", "semana", "pernoctaciones", "visitantes"},
    "T4": {"ano", "mes", "cod_barrio", "origen", "tipo_visitante", "visitantes"},
    "T5": {"ano", "mes", "date_dt", "cod_barrio", "visitantes"},
    "T6": {"ano", "mes", "cod_barrio", "cod_municipio_residencia", "visitantes"},
    "T7": {"ano", "mes", "cod_barrio", "provincia_residencia", "visitantes"},
    "T8": {"ano", "mes", "cod_barrio", "pais_residencia", "visitantes"},
    "T9": {"ano", "mes", "cod_barrio", "genero", "edad", "nivel_renta", "visitantes"},
}
for key, expected in required.items():
    if key in tables:
        missing = expected - set(tables[key].columns)
        print(key, "OK" if not missing else f"faltan {sorted(missing)}")

for key in ["T1", "T2", "T3", "T4", "T5"]:
    if key in tables:
        tables[key]["visitantes"] = pd.to_numeric(tables[key]["visitantes"], errors="coerce")
        print(key, "visitantes negativos:", int((tables[key]["visitantes"] < 0).sum()))


## 4. Evolución mensual municipal (T1)

In [ ]:
if "T1" in tables:
    t1 = tables["T1"].copy()
    t1["periodo"] = t1["mes"].astype(str)
    mensual = (t1.groupby(["periodo", "origen", "tipo_visitante"], dropna=False, as_index=False)["visitantes"]
                 .sum())
    pivot = mensual.pivot_table(index="periodo", columns="tipo_visitante", values="visitantes", aggfunc="sum")
    pivot.plot(figsize=(13, 4), marker="o", title="T1 — visitantes por mes y tipo")
    plt.ylabel("Visitantes (agregación publicada)")
    plt.xlabel("Mes")
    plt.xticks(rotation=45)
    plt.show()
    display(mensual.tail(12))


## 5. Picos diarios (T2)

In [ ]:
if "T2" in tables:
    t2 = tables["T2"].copy()
    t2["date_dt"] = pd.to_datetime(t2["date_dt"], errors="coerce")
    top_days = (t2.groupby("date_dt", as_index=False)["visitantes"].sum()
                  .sort_values("visitantes", ascending=False).head(15))
    display(top_days)
    top_days.sort_values("date_dt").plot(x="date_dt", y="visitantes", kind="bar", figsize=(13, 4), legend=False, title="T2 — días con más visitantes")
    plt.ylabel("Visitantes")
    plt.xlabel("Día")
    plt.show()


## 6. Barrios con mayor volumen mensual (T4)

In [ ]:
if "T4" in tables:
    t4 = tables["T4"].copy()
    ranking = (t4.groupby("cod_barrio", as_index=False)["visitantes"].sum()
                 .sort_values("visitantes", ascending=False).head(20))
    display(ranking)
    ranking.sort_values("visitantes").plot.barh(x="cod_barrio", y="visitantes", legend=False, figsize=(8, 7), title="T4 — barrios con mayor volumen publicado")
    plt.xlabel("Visitantes")
    plt.ylabel("Código de barrio")
    plt.show()


## 7. Procedencia y perfil

Estas tablas sirven para comparar distribuciones, pero no para sumar visitantes entre categorías como si fueran personas distintas.

In [ ]:
def show_distribution(key, dimension, n=15):
    if key not in tables or dimension not in tables[key]:
        return
    result = (tables[key].groupby(dimension, dropna=False, as_index=False)["visitantes"]
                .sum().sort_values("visitantes", ascending=False).head(n))
    display(result)

show_distribution("T6", "municipio_residencia")
show_distribution("T7", "provincia_residencia")
show_distribution("T8", "pais_residencia")
show_distribution("T9", "nivel_renta")


## 8. Exportar resultados de prueba

Los resultados derivados se guardan en `data/processed/`, nunca sobre los originales.

In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)
if "T4" in tables:
    ranking.to_csv(PROCESSED / "turismo_ranking_barrios_prueba.csv", index=False)
    print("Guardado:", PROCESSED / "turismo_ranking_barrios_prueba.csv")
